In [1]:
import numpy as np

In [46]:
X = np.array([1,2,3]).astype(np.float32)

with open("logtest.spire", 'wb') as f:
    f.write(X.size.to_bytes(4,'big'))
    f.write(X.tobytes())


In [ ]:

f = open("logtest.spire", 'rb')
n_neurons = int.from_bytes(f.read(4), "big")
np.frombuffer(f.read(), dtype=np.float32).reshape(-1, n_neurons)




In [ ]:
import numpy as np
import os
import math
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib

filename = "./cuda_test_9.spire"
record_stride = 1  # set to the stride used when recording

def play_spire_recording(filename: str, save = False, record_stride: int = 1):
    matplotlib.use('TkAgg')

    if filename == '':
        return

    if record_stride < 1:
        raise ValueError("record_stride must be >= 1.")
    
    delimited = filename.split('.')
    if delimited[-1].lower() != "spire":
        print("Cannot parse non-SpiRe file.")
        return

    filesize_bytes = os.path.getsize(filename)

    try: 
        with open(filename, "rb") as f:
            # first 4 bytes = neuron count
            neuron_count = int.from_bytes(f.read(4), byteorder='big')
            f32_bytesize = 4
            tickdata_size = neuron_count * f32_bytesize
            sim_data_size = filesize_bytes - 4
            recording_length = int(sim_data_size / tickdata_size)
            print(sim_data_size, tickdata_size, recording_length)
            recording = np.zeros((recording_length, neuron_count), dtype=np.float32)

            tick_bytes = f.read(tickdata_size)
            tick = 0
            while tick_bytes:
                tick_data = np.frombuffer(tick_bytes, dtype=np.float32, count=neuron_count)
                recording[tick] += tick_data
                tick+=1
                tick_bytes = f.read(tickdata_size)

    except Exception as e:
        print(e)
        return

    N = int(math.sqrt(neuron_count))
    print(recording.shape)
    fig, ax = plt.subplots()
    im = ax.imshow(recording[0].reshape(N, N), cmap='viridis', aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.set_title(f'Tick : {0 * record_stride}')

    def update(frame):
        im.set_array(recording[frame].reshape(N, N))
        ax.set_title(f'Tick : {frame * record_stride}')
        return [im]

    anim = animation.FuncAnimation(
        fig, 
        update, 
        frames=recording_length,
        interval=1,
        blit=True,
        repeat=False
    )

    if save: 
        anim.save(f'{delimited[0]}.mp4', writer='ffmpeg', fps=60)
    else: 
        plt.show()


play_spire_recording(filename, save=False, record_stride=record_stride)





2621440000 262144 10000
(10000, 65536)


In [ ]:
X.size